In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROCESSED_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PROJECT_ROOT / "results" / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


ModuleNotFoundError: No module named 'torch'

In [2]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

class_names = sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()])
assert len(class_names) == 38, f"Expected 38 classes, found {len(class_names)}"

class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

rows = []
for cls in class_names:
    cls_dir = RAW_DIR / cls
    for fp in cls_dir.iterdir():
        if fp.is_file() and fp.suffix.lower() in IMG_EXTS:
            rows.append({"filepath": str(fp), "label": class_to_idx[cls], "class_name": cls})

df = pd.DataFrame(rows)
print("Total images:", len(df))
df.head()


NameError: name 'RAW_DIR' is not defined

In [3]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

class_names = sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()])
assert len(class_names) == 38, f"Expected 38 classes, found {len(class_names)}"

class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

rows = []
for cls in class_names:
    cls_dir = RAW_DIR / cls
    for fp in cls_dir.iterdir():
        if fp.is_file() and fp.suffix.lower() in IMG_EXTS:
            rows.append({"filepath": str(fp), "label": class_to_idx[cls], "class_name": cls})

df = pd.DataFrame(rows)
print("Total images:", len(df))
df.head()

NameError: name 'RAW_DIR' is not defined

In [4]:
IMG_SIZE = 224

train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ToTensor(),  # scales to [0,1]
])

val_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])


NameError: name 'T' is not defined

In [5]:
class PlantVillageDataset(Dataset):
    def __init__(self, csv_path: Path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

train_ds = PlantVillageDataset(train_path, transform=train_tfms)
val_ds   = PlantVillageDataset(val_path, transform=val_tfms)
test_ds  = PlantVillageDataset(test_path, transform=val_tfms)

len(train_ds), len(val_ds), len(test_ds)


NameError: name 'Dataset' is not defined

In [6]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


NameError: name 'DataLoader' is not defined

In [7]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = BaselineCNN(num_classes=38).to(DEVICE)
model


NameError: name 'nn' is not defined

In [8]:
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    criterion = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        with torch.set_grad_enabled(train_mode):
            logits = model(x)
            loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    avg_loss = total_loss / total
    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    macro_recall = recall_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, macro_f1, macro_recall

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 8

history = []
best_val_f1 = -1.0
best_path = PROJECT_ROOT / "models" / "baseline_cnn_best.pt"
best_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1, tr_rec = run_epoch(model, train_loader, optimizer)
    va_loss, va_acc, va_f1, va_rec = run_epoch(model, val_loader, optimizer=None)

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss, "train_acc": tr_acc, "train_macro_f1": tr_f1, "train_macro_recall": tr_rec,
        "val_loss": va_loss, "val_acc": va_acc, "val_macro_f1": va_f1, "val_macro_recall": va_rec
    })

    print(f"Epoch {epoch:02d} | "
          f"TR loss {tr_loss:.4f} acc {tr_acc:.3f} f1 {tr_f1:.3f} | "
          f"VA loss {va_loss:.4f} acc {va_acc:.3f} f1 {va_f1:.3f}")

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        torch.save(model.state_dict(), best_path)

best_path, best_val_f1


NameError: name 'torch' is not defined

In [9]:
hist = pd.DataFrame(history)
hist.to_csv(RESULTS_DIR / "training_log.csv", index=False)

plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train")
plt.plot(hist["epoch"], hist["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend()
plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=200)
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["train_macro_f1"], label="train_macro_f1")
plt.plot(hist["epoch"], hist["val_macro_f1"], label="val_macro_f1")
plt.title("Macro-F1")
plt.xlabel("Epoch"); plt.ylabel("Macro-F1")
plt.legend()
plt.savefig(RESULTS_DIR / "macro_f1_curve.png", dpi=200)
plt.show()


NameError: name 'history' is not defined

In [10]:
# Load best baseline checkpoint
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

test_acc = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average="macro")
test_macro_recall = recall_score(y_true, y_pred, average="macro")

print("TEST accuracy:", test_acc)
print("TEST macro-F1 :", test_macro_f1)
print("TEST macro-recall:", test_macro_recall)


NameError: name 'model' is not defined

In [11]:
# Per-class metrics table
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).T
df_report.to_csv(RESULTS_DIR / "classification_report.csv")
df_report.head()


NameError: name 'y_true' is not defined

In [12]:
# Confusion matrix (saved)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation="nearest")
plt.title("Baseline CNN Confusion Matrix (Test)")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200)
plt.show()


NameError: name 'y_true' is not defined